<a href="https://colab.research.google.com/github/anushkasingru/Time_Series_Analysis_most_common_models/blob/main/Time_Series_Analysis_most_common_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# =============================================================================
# Interactive Time Series Lab (Google Colab Anchored Leaderboard Architecture)
# =============================================================================

# ---------- Install dependencies ----------
!pip -q install statsmodels openpyxl prophet --upgrade >/dev/null 2>&1 || true

import io, sys, warnings, os, math
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew

import ipywidgets as widgets
from ipywidgets import HTML as WHTML  # Widget HTML
from IPython.display import display, clear_output, HTML as DHTML  # Display HTML

# Enable Colab custom widget manager
from google.colab import output, files as colab_files
output.enable_custom_widget_manager()

# time-series / stats
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.api import VAR

# Prophet optional
prophet_available = False
try:
    from prophet import Prophet
    prophet_available = True
except Exception:
    try:
        !pip -q install prophet >/dev/null 2>&1
        from prophet import Prophet
        prophet_available = True
    except Exception:
        prophet_available = False

# ---------------- Global state ----------------
df = None
date_col = None
selected_vars = []
results_log = []

# Top-level flat output containers
step_out = widgets.Output()         # Step navigation & model controls (Top)
log_out = widgets.Output()          # Test & model execution plots (Middle)
leaderboard_out = widgets.Output()  # Live Leaderboard (Anchored Bottom)

# ---------------- Helpers ----------------
def numeric_columns(df_local):
    return df_local.select_dtypes(include=[np.number]).columns.tolist()

def compute_metrics(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    if y_true.shape != y_pred.shape:
        n = min(len(y_true), len(y_pred))
        y_true = y_true[-n:]
        y_pred = y_pred[-n:]
    mae = np.mean(np.abs(y_true - y_pred))
    rm = np.sqrt(np.mean((y_true - y_pred) ** 2))
    denom = np.where(y_true == 0, np.nan, y_true)
    mape = np.mean(np.abs((y_true - y_pred) / denom)) * 100
    mape = float('nan') if np.isnan(mape) else mape
    return {'RMSE': rm, 'MAE': mae, 'MAPE': mape}

def download_dataframe(df_obj, fname='output.csv'):
    df_obj.to_csv(fname, index=False)
    try:
        colab_files.download(fname)
    except Exception:
        display(DHTML(f"<a href='{fname}' download>Download {fname}</a>"))

# ---------------- Step 1: Native Upload ----------------
def start_lab():
    global df
    clear_output()
    print("📌 Interactive Time Series Lab — Ready")
    print("Please select your CSV or Excel file below:\n")

    uploaded = colab_files.upload()
    if not uploaded:
        print("❌ No file uploaded.")
        return

    fname = list(uploaded.keys())[0]
    content = uploaded[fname]

    try:
        if fname.lower().endswith('.csv'):
            try:
                df = pd.read_csv(io.BytesIO(content))
            except Exception:
                df = pd.read_csv(io.BytesIO(content), encoding='latin1')
        else:
            df = pd.read_excel(io.BytesIO(content))
    except Exception as e:
        print("❌ Failed to parse file:", e)
        return

    print(f"\n✅ File '{fname}' loaded successfully ({df.shape[0]} rows, {df.shape[1]} columns).")
    display(df.head(5))

    # Render layout hierarchy (Top -> Middle -> Bottom)
    display(step_out)
    display(log_out)
    display(leaderboard_out)

    update_leaderboard()
    choose_date_column()

# ---------------- Step 2: Date Parsing ----------------
def choose_date_column():
    step_out.clear_output()
    with step_out:
        display(DHTML("<h3>Step 1: Choose Date Column & Format</h3>"))

        date_dropdown = widgets.Dropdown(options=list(df.columns), description='Date column:')
        date_format = widgets.Dropdown(
            options=['dd/mm/yyyy', 'dd-mm-yyyy', 'mm-yyyy', 'yyyy', 'auto'],
            value='dd/mm/yyyy',
            description='Format:'
        )
        next_btn = widgets.Button(description='Set Date & Continue', button_style='success')

        def on_set(b):
            global date_col, df
            date_col = date_dropdown.value
            fmt = date_format.value
            try:
                if fmt == 'dd/mm/yyyy':
                    df[date_col] = pd.to_datetime(df[date_col], format='%d/%m/%Y', errors='coerce', dayfirst=True)
                elif fmt == 'dd-mm-yyyy':
                    df[date_col] = pd.to_datetime(df[date_col], format='%d-%m-%Y', errors='coerce', dayfirst=True)
                elif fmt == 'mm-yyyy':
                    df[date_col] = pd.to_datetime(df[date_col], format='%m-%Y', errors='coerce')
                    df[date_col] = df[date_col].dt.to_period('M').dt.to_timestamp()
                elif fmt == 'yyyy':
                    df[date_col] = pd.to_datetime(df[date_col], format='%Y', errors='coerce')
                    df[date_col] = df[date_col].dt.to_period('Y').dt.to_timestamp()
                else:
                    df[date_col] = pd.to_datetime(df[date_col], errors='coerce', dayfirst=True)

                df = df.dropna(subset=[date_col]).copy()
                df.sort_values(by=date_col, inplace=True)
                df.set_index(date_col, inplace=True)
                print("✅ Date set as index successfully.")
            except Exception as e:
                print("❌ Date parse failed:", e)
                return

            choose_variables()

        display(widgets.VBox([date_dropdown, date_format, next_btn]))
        next_btn.on_click(on_set)

# ---------------- Step 3: Variables Selection ----------------
def choose_variables():
    step_out.clear_output()
    with step_out:
        display(DHTML("<h3>Step 2: Choose Variables</h3>"))

        num_cols = numeric_columns(df)
        if not num_cols:
            print("❌ No numeric columns found.")
            return

        multi_select = widgets.SelectMultiple(
            options=num_cols,
            description='Variables:',
            disabled=False,
            layout={'height': '150px'}
        )

        proceed = widgets.Button(description='Proceed → Missing Analysis', button_style='primary')

        def on_proceed(b):
            global selected_vars
            selected_vars = list(multi_select.value)
            if not selected_vars:
                print("⚠️ Select at least one variable.")
                return
            missing_value_analysis()

        display(widgets.VBox([multi_select, proceed]))
        proceed.on_click(on_proceed)

# ---------------- Step 4: Missing Values ----------------
def missing_value_analysis():
    step_out.clear_output()
    with step_out:
        display(DHTML("<h3>Step 3: Missing Value Analysis & Imputation</h3>"))
        available = [c for c in selected_vars if c in df.columns]

        miss = df[available].isna().sum().to_frame('Missing_Count')
        miss['Missing_%'] = (miss['Missing_Count'] / len(df) * 100).round(2)
        display(miss)

        impute_btn = widgets.Button(description='Auto-impute & Continue', button_style='primary')

        def do_auto(b):
            for c in available:
                if miss.loc[c, 'Missing_%'] > 0:
                    col_skew = float(skew(df[c].dropna())) if df[c].dropna().shape[0] > 0 else 0.0
                    if math.isfinite(col_skew) and abs(col_skew) < 1:
                        df[c] = df[c].fillna(df[c].mean())
                    else:
                        df[c] = df[c].fillna(df[c].median())
            descriptive_and_corr()

        display(widgets.VBox([impute_btn]))
        impute_btn.on_click(do_auto)

# ---------------- Step 5: Descriptives ----------------
def descriptive_and_corr():
    step_out.clear_output()
    with step_out:
        display(DHTML("<h3>Step 4: Descriptive Statistics & Correlation</h3>"))
        vars_here = [c for c in selected_vars if c in df.columns]

        display(df[vars_here].describe().T)

        plt.figure(figsize=(max(6, len(vars_here)), 4))
        sns.heatmap(df[vars_here].corr(), annot=True, cmap='coolwarm', fmt='.2f')
        plt.title("Correlation Heatmap")
        plt.show()

        diagnostics_menu()

# ---------------- Step 6: Diagnostics Menu ----------------
def diagnostics_menu():
    step_out.clear_output()
    with step_out:
        display(DHTML("<h3>Step 5: Diagnostics — Run Tests</h3>"))
        display(DHTML("<p>Click any button below. Results append persistently in the Output Log below.</p>"))

        adf_b = widgets.Button(description='ADF Test', button_style='info')
        kpss_b = widgets.Button(description='KPSS Test', button_style='info')
        acf_pacf_b = widgets.Button(description='ACF / PACF Plots', button_style='info')
        decomp_b = widgets.Button(description='Seasonal Decompose', button_style='info')
        lb_b = widgets.Button(description='Ljung-Box Test', button_style='info')

        clear_log_b = widgets.Button(description='Clear Log Window', button_style='warning')
        cont = widgets.Button(description='Continue → Models', button_style='success')

        period_box = widgets.BoundedIntText(value=12, min=2, max=365, description='Period:')

        display(widgets.VBox([
            widgets.HBox([adf_b, kpss_b, acf_pacf_b, lb_b]),
            widgets.HBox([decomp_b, period_box]),
            WHTML("<br>"),
            widgets.HBox([clear_log_b, cont])
        ]))

        adf_b.on_click(run_adf)
        kpss_b.on_click(run_kpss)
        acf_pacf_b.on_click(run_acf_pacf)
        lb_b.on_click(run_ljung_box)
        decomp_b.on_click(lambda b: run_decomposition(period_box.value))
        clear_log_b.on_click(lambda b: log_out.clear_output())
        cont.on_click(lambda b: models_menu())

def run_adf(b):
    with log_out:
        display(DHTML("<h4>📌 Augmented Dickey-Fuller (ADF) Test Results</h4>"))
        rows = []
        for c in selected_vars:
            s = df[c].dropna()
            try:
                stat, p, _, _, _, _ = adfuller(s)
                rows.append([c, round(stat, 4), round(p, 4), 'Stationary' if p < 0.05 else 'Not stationary (Unit Root)'])
            except Exception as e:
                rows.append([c, np.nan, np.nan, f'Error: {e}'])
        display(pd.DataFrame(rows, columns=['Variable', 'Test Statistic', 'p-value', 'Verdict']))
        display(DHTML("<hr>"))

def run_kpss(b):
    with log_out:
        display(DHTML("<h4>📌 KPSS Test Results</h4>"))
        rows = []
        for c in selected_vars:
            s = df[c].dropna()
            try:
                stat, p, _, _ = kpss(s, nlags='auto')
                rows.append([c, round(stat, 4), round(p, 4), 'Stationary' if p > 0.05 else 'Not stationary'])
            except Exception as e:
                rows.append([c, np.nan, np.nan, f'Error: {e}'])
        display(pd.DataFrame(rows, columns=['Variable', 'Test Statistic', 'p-value', 'Verdict']))
        display(DHTML("<hr>"))

def run_acf_pacf(b):
    with log_out:
        display(DHTML("<h4>📌 Autocorrelation (ACF) & Partial Autocorrelation (PACF) Plots</h4>"))
        for c in selected_vars:
            s = df[c].dropna()
            max_lags = max(1, min(24, len(s) // 2 - 1))
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
            plot_acf(s, ax=ax1, lags=max_lags)
            ax1.set_title(f'ACF: {c}')
            plot_pacf(s, ax=ax2, lags=max_lags, method='ywm')
            ax2.set_title(f'PACF: {c}')
            plt.tight_layout()
            plt.show()
        display(DHTML("<hr>"))

def run_decomposition(period):
    with log_out:
        display(DHTML(f"<h4>📌 Seasonal Decomposition Results (Period = {period})</h4>"))
        for c in selected_vars:
            s = df[c].dropna()
            freq = pd.infer_freq(df.index)
            try:
                if freq:
                    s_freq = s.asfreq(freq)
                    dec = seasonal_decompose(s_freq, model='additive', period=period)
                else:
                    dec = seasonal_decompose(s, model='additive', period=period)
                fig = dec.plot()
                fig.suptitle(f"Decomposition: {c}", y=1.02)
                plt.show()
            except Exception as e:
                print("Decomposition failed for", c, ":", e)
        display(DHTML("<hr>"))

def run_ljung_box(b):
    with log_out:
        display(DHTML("<h4>📌 Ljung-Box Test Results (Lag 10)</h4>"))
        rows = []
        for c in selected_vars:
            s = df[c].dropna()
            try:
                lb_df = acorr_ljungbox(s, lags=[10], return_df=True)
                stat = lb_df['lb_stat'].iloc[0]
                p = lb_df['lb_pvalue'].iloc[0]
                rows.append([c, round(stat, 4), round(p, 4), 'No autocorrelation' if p > 0.05 else 'Autocorrelation present'])
            except Exception as e:
                rows.append([c, np.nan, np.nan, f'Error: {e}'])
        display(pd.DataFrame(rows, columns=['Variable', 'Test Statistic', 'p-value', 'Verdict']))
        display(DHTML("<hr>"))

# ---------------- Step 7: Models Menu ----------------
def models_menu():
    step_out.clear_output()
    with step_out:
        display(DHTML("<h3>Step 6: Models — Configure & Run</h3>"))
        display(DHTML("<p>Set model parameters below and click <b>Run Selected Models</b>. Controls stay visible, execution details log in the middle, and the leaderboard updates at the bottom.</p>"))

        # All model checkboxes default to True
        arima_cb = widgets.Checkbox(value=True, description='ARIMA')
        sarima_cb = widgets.Checkbox(value=True, description='SARIMA')
        hw_cb = widgets.Checkbox(value=True, description='Holt-Winters')
        var_cb = widgets.Checkbox(value=True, description='VAR (multivariate)')
        prophet_cb = widgets.Checkbox(value=prophet_available, description='Prophet', disabled=not prophet_available)

        active_cols = [c for c in selected_vars if c in df.columns]

        # ARIMA params
        arima_target = widgets.Dropdown(options=active_cols, description='ARIMA target:')
        arima_p = widgets.BoundedIntText(value=1, min=0, max=6, description='p:')
        arima_d = widgets.BoundedIntText(value=0, min=0, max=3, description='d:')
        arima_q = widgets.BoundedIntText(value=1, min=0, max=6, description='q:')

        # SARIMA params
        sarima_target = widgets.Dropdown(options=active_cols, description='SARIMA target:')
        sarima_P = widgets.BoundedIntText(value=0, min=0, max=4, description='P:')
        sarima_D = widgets.BoundedIntText(value=0, min=0, max=2, description='D:')
        sarima_Q = widgets.BoundedIntText(value=0, min=0, max=4, description='Q:')
        sarima_s = widgets.BoundedIntText(value=12, min=1, max=365, description='seasonal s:')

        # Holt-Winters params
        hw_target = widgets.Dropdown(options=active_cols, description='HW target:')
        hw_trend = widgets.Dropdown(options=[None, 'add', 'mul'], description='Trend:')
        hw_seasonal = widgets.Dropdown(options=[None, 'add', 'mul'], description='Seasonal:')
        hw_sp = widgets.BoundedIntText(value=12, min=1, max=365, description='sp:')

        # VAR params
        var_lags = widgets.BoundedIntText(value=1, min=1, max=12, description='VAR Lags:')

        # Prophet params
        prophet_target = widgets.Dropdown(options=active_cols, description='Prophet target:')

        fh_in = widgets.BoundedIntText(value=12, min=1, max=240, description='Horizon:')

        run_btn = widgets.Button(description='Run Selected Models', button_style='primary')
        clear_log_b = widgets.Button(description='Clear Log Window', button_style='warning')

        display(widgets.VBox([
            WHTML("<b>Select Models to Include:</b>"),
            widgets.HBox([arima_cb, sarima_cb, hw_cb, var_cb, prophet_cb]),
            WHTML("<hr>"),
            WHTML("<b>ARIMA Settings:</b>"),
            widgets.HBox([arima_target, arima_p, arima_d, arima_q]),
            WHTML("<hr>"),
            WHTML("<b>SARIMA Settings:</b>"),
            widgets.HBox([sarima_target, sarima_P, sarima_D, sarima_Q, sarima_s]),
            WHTML("<hr>"),
            WHTML("<b>Holt-Winters Settings:</b>"),
            widgets.HBox([hw_target, hw_trend, hw_seasonal, hw_sp]),
            WHTML("<hr>"),
            WHTML("<b>VAR Settings:</b>"),
            var_lags,
            WHTML("<hr>"),
            WHTML("<b>Prophet Settings:</b>"),
            prophet_target,
            WHTML("<hr>"),
            fh_in,
            WHTML("<br>"),
            widgets.HBox([run_btn, clear_log_b])
        ]))

        def on_run(b):
            with log_out:
                display(DHTML("<h2>🚀 Model Execution Output Log</h2>"))
                fh = fh_in.value

                if arima_cb.value:
                    run_arima_model(arima_target.value, arima_p.value, arima_d.value, arima_q.value, fh)
                if sarima_cb.value:
                    run_sarima_model(sarima_target.value, arima_p.value, arima_d.value, arima_q.value,
                                     sarima_P.value, sarima_D.value, sarima_Q.value, sarima_s.value, fh)
                if hw_cb.value:
                    run_hw_model(hw_target.value, hw_trend.value, hw_seasonal.value, hw_sp.value, fh)
                if var_cb.value:
                    run_var_model(active_cols, var_lags.value, fh)
                if prophet_cb.value and prophet_available:
                    run_prophet_model(prophet_target.value, fh)

                display(DHTML("<hr style='border: 2px solid #333;'>"))

            # Update live leaderboard anchored at the bottom
            update_leaderboard()

        run_btn.on_click(on_run)
        clear_log_b.on_click(lambda b: log_out.clear_output())

# ---------------- Model Executors ----------------
def run_arima_model(target, p, d, q, fh):
    y = df[target].dropna()
    print(f"\n▶ Running ARIMA({p},{d},{q}) on {target}...")
    try:
        model = ARIMA(y, order=(p, d, q))
        res = model.fit()
        fc_res = res.get_forecast(steps=fh)
        fc = fc_res.predicted_mean

        plt.figure(figsize=(10, 4))
        plt.plot(y.index, y.values, label='Actual')
        plt.plot(res.fittedvalues.index, res.fittedvalues.values, label='Fitted')
        plt.plot(fc.index, fc.values, label='Forecast', color='red')
        plt.title(f"ARIMA({p},{d},{q}) — {target}")
        plt.legend()
        plt.show()

        metrics = compute_metrics(y.values, res.fittedvalues.values)
        entry = {'Model': 'ARIMA', 'Target': target, 'Params': f'({p},{d},{q})', 'RMSE': metrics['RMSE'], 'MAE': metrics['MAE'], 'MAPE': metrics['MAPE'], 'AIC': res.aic}
        results_log.append(entry)
        display(pd.DataFrame([entry]))
    except Exception as e:
        print("ARIMA failed:", e)

def run_sarima_model(target, p, d, q, P, D, Q, s, fh):
    y = df[target].dropna()
    print(f"\n▶ Running SARIMA({p},{d},{q})x({P},{D},{Q})[{s}] on {target}...")
    try:
        model = ARIMA(y, order=(p, d, q), seasonal_order=(P, D, Q, s))
        res = model.fit()
        fc_res = res.get_forecast(steps=fh)
        fc = fc_res.predicted_mean

        plt.figure(figsize=(10, 4))
        plt.plot(y.index, y.values, label='Actual')
        plt.plot(res.fittedvalues.index, res.fittedvalues.values, label='Fitted')
        plt.plot(fc.index, fc.values, label='Forecast', color='red')
        plt.title(f"SARIMA({p},{d},{q})x({P},{D},{Q})[{s}] — {target}")
        plt.legend()
        plt.show()

        metrics = compute_metrics(y.values, res.fittedvalues.values)
        entry = {'Model': 'SARIMA', 'Target': target, 'Params': f'({p},{d},{q})x({P},{D},{Q})[{s}]', 'RMSE': metrics['RMSE'], 'MAE': metrics['MAE'], 'MAPE': metrics['MAPE'], 'AIC': res.aic}
        results_log.append(entry)
        display(pd.DataFrame([entry]))
    except Exception as e:
        print("SARIMA failed:", e)

def run_hw_model(target, trend, seasonal, sp, fh):
    y = df[target].dropna()
    print(f"\n▶ Running Holt-Winters on {target} (trend={trend}, seasonal={seasonal}, sp={sp})...")
    try:
        model = ExponentialSmoothing(y, trend=trend, seasonal=seasonal, seasonal_periods=sp)
        res = model.fit()
        fc = res.forecast(fh)

        plt.figure(figsize=(10, 4))
        plt.plot(y.index, y.values, label='Actual')
        plt.plot(res.fittedvalues.index, res.fittedvalues.values, label='Fitted')
        plt.plot(fc.index, fc.values, label='Forecast', color='red')
        plt.title(f"Holt-Winters — {target}")
        plt.legend()
        plt.show()

        metrics = compute_metrics(y.values, res.fittedvalues.values)
        entry = {'Model': 'Holt-Winters', 'Target': target, 'Params': f'trend={trend},seasonal={seasonal},sp={sp}', 'RMSE': metrics['RMSE'], 'MAE': metrics['MAE'], 'MAPE': metrics['MAPE'], 'AIC': np.nan}
        results_log.append(entry)
        display(pd.DataFrame([entry]))
    except Exception as e:
        print("Holt-Winters failed:", e)

def run_var_model(vars_list, k, fh):
    if len(vars_list) < 2:
        print("⚠️ VAR requires at least 2 numeric variables selected.")
        return
    Y = df[vars_list].dropna()
    print(f"\n▶ Running VAR(lag={k}) on [{', '.join(vars_list)}]...")
    try:
        model = VAR(Y)
        res = model.fit(k)
        fc = res.forecast(y=Y.values[-k:], steps=fh)
        fc_df = pd.DataFrame(fc, columns=Y.columns)

        for col in vars_list:
            plt.figure(figsize=(10, 3))
            plt.plot(Y.index, Y[col], label='Actual')
            plt.plot(res.fittedvalues.index, res.fittedvalues[col], label='Fitted')
            plt.title(f"VAR — {col}")
            plt.legend()
            plt.show()

        entry = {'Model': 'VAR', 'Target': ','.join(vars_list), 'Params': f'lags={k}', 'RMSE': np.nan, 'MAE': np.nan, 'MAPE': np.nan, 'AIC': res.aic}
        results_log.append(entry)
        display(pd.DataFrame([entry]))
    except Exception as e:
        print("VAR failed:", e)

def run_prophet_model(target, fh):
    y = df[[target]].dropna().reset_index().rename(columns={date_col:'ds', target:'y'})
    print(f"\n▶ Running Prophet on {target}...")
    try:
        m = Prophet()
        m.fit(y)
        future = m.make_future_dataframe(periods=fh)
        forecast = m.predict(future)
        fig = m.plot(forecast)
        plt.title(f"Prophet Forecast — {target}")
        plt.show()

        in_sample = forecast[forecast['ds'] <= y['ds'].max()]
        y_pred = in_sample['yhat'].values[-len(y):]
        metrics = compute_metrics(y['y'].values, y_pred)

        entry = {'Model': 'Prophet', 'Target': target, 'Params': 'default', 'RMSE': metrics['RMSE'], 'MAE': metrics['MAE'], 'MAPE': metrics['MAPE'], 'AIC': np.nan}
        results_log.append(entry)
        display(pd.DataFrame([entry]))
    except Exception as e:
        print("Prophet failed:", e)

# ---------------- Leaderboard (Anchored Bottom) ----------------
def update_leaderboard():
    leaderboard_out.clear_output()
    with leaderboard_out:
        display(DHTML("<h3>📊 Model Leaderboard Summary (Live)</h3>"))
        if not results_log:
            display(DHTML("<p><i>No models executed yet. Configure parameters above and click <b>Run Selected Models</b> to populate this leaderboard.</i></p>"))
            return

        lb = pd.DataFrame(results_log)
        for col in ['RMSE', 'MAE', 'MAPE', 'AIC']:
            if col not in lb.columns:
                lb[col] = np.nan

        sort_col = widgets.Dropdown(
            options=[c for c in ['RMSE', 'MAE', 'MAPE', 'AIC'] if c in lb.columns],
            value='RMSE',
            description='Sort by:'
        )
        sort_order = widgets.Dropdown(
            options=[('Ascending (Low → High)', True), ('Descending (High → Low)', False)],
            value=True,
            description='Order:'
        )
        sort_btn = widgets.Button(description='Apply Sort', button_style='info')
        dl_btn = widgets.Button(description='Download CSV', button_style='success')

        table_whtml = WHTML()

        def render_table():
            col = sort_col.value
            asc = sort_order.value
            sorted_lb = lb.sort_values(by=col, ascending=asc, na_position='last').reset_index(drop=True)
            styled_df = sorted_lb.copy()
            for numeric_c in ['RMSE', 'MAE', 'MAPE', 'AIC']:
                if numeric_c in styled_df.columns:
                    styled_df[numeric_c] = styled_df[numeric_c].apply(lambda x: f"{x:.4f}" if pd.notnull(x) else "N/A")
            table_whtml.value = styled_df.to_html(index=False, classes="dataframe")

        def on_sort_click(b):
            render_table()

        def do_dl(b):
            col = sort_col.value
            asc = sort_order.value
            sorted_lb = lb.sort_values(by=col, ascending=asc, na_position='last').reset_index(drop=True)
            download_dataframe(sorted_lb, 'model_leaderboard.csv')

        sort_btn.on_click(on_sort_click)
        dl_btn.on_click(do_dl)

        render_table()

        display(widgets.VBox([
            widgets.HBox([sort_col, sort_order, sort_btn, dl_btn]),
            WHTML("<br>"),
            table_whtml
        ]))

# Execute Pipeline
start_lab()

📌 Interactive Time Series Lab — Ready
Please select your CSV or Excel file below:



Saving walmart_final_timeseries.csv to walmart_final_timeseries.csv

✅ File 'walmart_final_timeseries.csv' loaded successfully (143 rows, 2 columns).


,Date,Weekly_Sales
0,2010-02-05,49750875.98
1,2010-02-12,48336800.10
2,2010-02-19,48277902.33
3,2010-02-26,43970440.65
4,2010-03-05,46872715.16


Output()

Output()

Output()

✅ Date set as index successfully.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>